# 🔢 Aula 2 — Programação Inteira e Otimização Combinatória
**Pesquisa Operacional e Otimização em IA** — MBA em Ciência de Dados (UNIFOR)

Prof. Mafra | mafra@verboo.ai

---

Hoje a aula é **toda no Python**. Vamos construir e resolver problemas juntos, ao vivo.

In [20]:
!pip install pulp -q
from pulp import *


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


---
## 1. O problema da Fábrica de Móveis

Uma fábrica produz **cadeiras** e **mesas**. Cada cadeira dá R$ 40 de lucro e usa 2 m² de madeira e 3h de mão de obra. Cada mesa dá R$ 70 e usa 5 m² e 4h. A fábrica tem 30 m² de madeira e 28h de mão de obra.

**Pergunta pra vocês:** quantas cadeiras e mesas fabricar pra maximizar o lucro?

In [21]:
# Fábrica de Móveis — resultado da PL

prob = LpProblem("Fabrica_Moveis", LpMaximize)

x1 = LpVariable("cadeiras", lowBound=0)
x2 = LpVariable("mesas", lowBound=0)

prob += 40*x1 + 70*x2, "Lucro"
prob += 2*x1 + 5*x2 <= 30, "Madeira"
prob += 3*x1 + 4*x2 <= 28, "Mao_de_obra"

prob.solve(PULP_CBC_CMD(msg=0))

print("=" * 40)
print("  RESULTADO — Fábrica de Móveis (PL)")
print("=" * 40)
print(f"  Cadeiras: {x1.varValue:.4f}")
print(f"  Mesas:    {x2.varValue:.4f}")
print(f"  Lucro:    R$ {value(prob.objective):.2f}")
print("=" * 40)
print(f"\n🤔 {x1.varValue:.4f} cadeiras? {x2.varValue:.4f} mesas?")
print(f"   Como fabricar pedaço de móvel?")

  RESULTADO — Fábrica de Móveis (PL)
  Cadeiras: 2.8571
  Mesas:    4.8571
  Lucro:    R$ 454.29

🤔 2.8571 cadeiras? 4.8571 mesas?
   Como fabricar pedaço de móvel?


### O que vocês fariam?

O resultado deu 2.86 cadeiras e 4.86 mesas. Na vida real, não dá pra fabricar 0.86 de uma mesa.

**Opções:**
- Arredondar pra baixo? (2 cadeiras, 4 mesas)
- Arredondar pra cima? (3 cadeiras, 5 mesas)
- Alguma outra combinação?

Vamos testar:

In [26]:
# Testando os arredondamentos

cenarios = [
    ("PL (fracionário)", 2.857, 4.857),
    ("Arredondar ↓ (2, 4)", 2, 4),
    ("Arredondar ↑ (3, 5)", 3, 5),
    ("Misto (3, 4)", 3, 4),
    ("Teste (2,5)", 2, 5),
    ("Teste 2 (4, 4)", 4, 4),
]

print(f"{'Cenário':<25} {'Cad':>5} {'Mesa':>5} {'Lucro':>10} {'Madeira':>10} {'MO':>8} {'Viável?':>8}")
print("-" * 75)
for nome, c, m in cenarios:
    lucro = 40*c + 70*m
    madeira = 2*c + 5*m
    mo = 3*c + 4*m
    ok_mad = madeira <= 30
    ok_mo = mo <= 28
    viavel = "✅" if ok_mad and ok_mo else "❌"
    if not ok_mad: viavel += " mad"
    if not ok_mo: viavel += " MO"
    print(f"{nome:<25} {c:>5.2f} {m:>5.2f} {'R$ '+str(int(lucro)):>10} {madeira:>10.1f} {mo:>8.1f} {viavel:>8}")

print(f"\n⚠️  Arredondar pra cima (3, 5): madeira = 31 > 30. INVIÁVEL!")
print(f"   Arredondar pra baixo (2, 4): viável, mas lucro = R$ 360 (perdeu R$ 94)")
print(f"   E (3, 4)? Lucro = R$ 400 e viável! Mas será que é o melhor?")
print(f"\n→ Vamos deixar o computador decidir...")

Cenário                     Cad  Mesa      Lucro    Madeira       MO  Viável?
---------------------------------------------------------------------------
PL (fracionário)           2.86  4.86     R$ 454       30.0     28.0        ✅
Arredondar ↓ (2, 4)        2.00  4.00     R$ 360       24.0     22.0        ✅
Arredondar ↑ (3, 5)        3.00  5.00     R$ 470       31.0     29.0 ❌ mad MO
Misto (3, 4)               3.00  4.00     R$ 400       26.0     25.0        ✅
Teste (2,5)                2.00  5.00     R$ 430       29.0     26.0        ✅
Teste 2 (4, 4)             4.00  4.00     R$ 440       28.0     28.0        ✅

⚠️  Arredondar pra cima (3, 5): madeira = 31 > 30. INVIÁVEL!
   Arredondar pra baixo (2, 4): viável, mas lucro = R$ 360 (perdeu R$ 94)
   E (3, 4)? Lucro = R$ 400 e viável! Mas será que é o melhor?

→ Vamos deixar o computador decidir...


---
## 2. Escalando o problema — Planos de IA (6 variáveis)

A fábrica tinha 2 variáveis. Fácil de visualizar. Mas e com 6?

Na Aula 1, vocês viram esse problema:

| Plano | Custo/mês (R$) | Produtividade (1-5) | Capacidade (1-10) |
|---|---|---|---|
| Codex Go | 45 | 3 | 2 |
| Codex Plus | 110 | 4 | 5 |
| Claude Pro | 110 | 4 | 7 |
| Claude Max | 550 | 5 | 10 |
| Gemini AI Plus | 25 | 3 | 3 |
| Gemini AI Pro | 97 | 4 | 8 |

**Resultado da PL:** Claude Max = **3.13**, Gemini Pro = **2.87**

Mesmo problema: resultado fracionário. Vamos resolver com PI e comparar.

In [4]:
# Primeiro, vamos rodar a versão FRACIONÁRIA (como na Aula 1)

prob_frac = LpProblem("Planos_IA_Fracionario", LpMaximize)

x1 = LpVariable("Codex_Go", lowBound=0)
x2 = LpVariable("Codex_Plus", lowBound=0)
x3 = LpVariable("Claude_Pro", lowBound=0)
x4 = LpVariable("Claude_Max", lowBound=0)
x5 = LpVariable("Gemini_Plus", lowBound=0)
x6 = LpVariable("Gemini_Pro", lowBound=0)

prob_frac += 3*x1 + 4*x2 + 4*x3 + 5*x4 + 3*x5 + 4*x6, "Produtividade"
prob_frac += 45*x1 + 110*x2 + 110*x3 + 550*x4 + 25*x5 + 97*x6 <= 2000, "Budget"
prob_frac += x1 + x2 + x3 + x4 + x5 + x6 == 6, "Pessoas"
prob_frac += 2*x1 + 5*x2 + 7*x3 + 10*x4 + 3*x5 + 8*x6 >= 30, "Capacidade"

prob_frac.solve(PULP_CBC_CMD(msg=0))

print("RESULTADO FRACIONÁRIO (Aula 1):")
print("=" * 40)
for v in prob_frac.variables():
    if v.varValue > 0.001:
        print(f"  {v.name}: {v.varValue:.4f}")
print(f"  Produtividade: {value(prob_frac.objective):.2f}")
print(f"\n❌ 3.13 licenças de Claude Max não existe!")

RESULTADO FRACIONÁRIO (Aula 1):
  Claude_Max: 3.1302
  Gemini_Pro: 2.8698
  Produtividade: 27.13

❌ 3.13 licenças de Claude Max não existe!


---
## 3. A mágica: `cat="Integer"`

Em PL, as variáveis podem ser qualquer número real (3.13, 2.87...).

Em **Programação Inteira (PI)**, forçamos as variáveis a serem números inteiros.

No PuLP, basta mudar **uma coisa**:

```python
# Antes (PL):
x = LpVariable("x", lowBound=0)

# Agora (PI):
x = LpVariable("x", lowBound=0, cat="Integer")
```

Só isso. Uma palavra muda tudo.

In [5]:
# MESMO problema, mas agora com cat="Integer"

prob_int = LpProblem("Planos_IA_Inteiro", LpMaximize)

x1 = LpVariable("Codex_Go", lowBound=0, cat="Integer")
x2 = LpVariable("Codex_Plus", lowBound=0, cat="Integer")
x3 = LpVariable("Claude_Pro", lowBound=0, cat="Integer")
x4 = LpVariable("Claude_Max", lowBound=0, cat="Integer")
x5 = LpVariable("Gemini_Plus", lowBound=0, cat="Integer")
x6 = LpVariable("Gemini_Pro", lowBound=0, cat="Integer")

prob_int += 3*x1 + 4*x2 + 4*x3 + 5*x4 + 3*x5 + 4*x6, "Produtividade"
prob_int += 45*x1 + 110*x2 + 110*x3 + 550*x4 + 25*x5 + 97*x6 <= 2000, "Budget"
prob_int += x1 + x2 + x3 + x4 + x5 + x6 == 6, "Pessoas"
prob_int += 2*x1 + 5*x2 + 7*x3 + 10*x4 + 3*x5 + 8*x6 >= 30, "Capacidade"

import time
t0 = time.time()
prob_int.solve(PULP_CBC_CMD(msg=0))
t1 = time.time()

print(f"⚙️  Solver: CBC (Branch-and-Bound)")
print(f"⚙️  Status: {LpStatus[prob_int.status]}")
print(f"⚙️  Tempo: {(t1-t0)*1000:.1f} ms")
print(f"⚙️  Variáveis: {len(prob_int.variables())}  |  Restrições: {len(prob_int.constraints)}")
print()
print("RESULTADO INTEIRO:")
print("=" * 40)
for v in prob_int.variables():
    if v.varValue > 0.001:
        print(f"  {v.name}: {v.varValue:.0f} licenças")
custo = 45*x1.varValue+110*x2.varValue+110*x3.varValue+550*x4.varValue+25*x5.varValue+97*x6.varValue
print(f"  Produtividade: {value(prob_int.objective):.0f}")
print(f"  Custo: R$ {custo:.0f} / 2000")
print(f"\n✅ Agora sim! Números inteiros que fazem sentido.")

⚙️  Solver: CBC (Branch-and-Bound)
⚙️  Status: Optimal
⚙️  Tempo: 22.9 ms
⚙️  Variáveis: 6  |  Restrições: 3

RESULTADO INTEIRO:
  Claude_Max: 3 licenças
  Gemini_Pro: 3 licenças
  Produtividade: 27
  Custo: R$ 1941 / 2000

✅ Agora sim! Números inteiros que fazem sentido.


In [6]:
# Comparação lado a lado

print(f"{'':>20} {'PL (fracionário)':>18} {'PI (inteiro)':>15}")
print("-" * 55)
print(f"{'Claude Max':>20} {'3.13':>18} {'3':>15}")
print(f"{'Gemini Pro':>20} {'2.87':>18} {'3':>15}")
print(f"{'Produtividade':>20} {'27.13':>18} {'27':>15}")
print(f"{'Custo':>20} {'R$ 2.000':>18} {'R$ 1.941':>15}")
print(f"\n💡 A PI deu produtividade um pouco menor (27 vs 27.13)")
print(f"   Isso SEMPRE acontece: restringir a inteiro nunca melhora.")
print(f"   O valor da PL é o 'teto' teórico — o melhor cenário possível.")
print(f"\n💡 E sobrou R$ 59 do budget! A PL usava os R$ 2.000 certinho.")

                       PL (fracionário)    PI (inteiro)
-------------------------------------------------------
          Claude Max               3.13               3
          Gemini Pro               2.87               3
       Produtividade              27.13              27
               Custo           R$ 2.000        R$ 1.941

💡 A PI deu produtividade um pouco menor (27 vs 27.13)
   Isso SEMPRE acontece: restringir a inteiro nunca melhora.
   O valor da PL é o 'teto' teórico — o melhor cenário possível.

💡 E sobrou R$ 59 do budget! A PL usava os R$ 2.000 certinho.


### E se arredondasse?

Lembram da Aula 1? "Arredonda pra 3 e 3 e pronto!" Nesse caso, coincidiu.

Mas e se arredondasse pra baixo (3 e 2)? Ou pra cima (4 e 3)?

In [38]:
# Cenários de arredondamento
cenarios = [
    ("PL (fracionário)", 3.13, 2.87),
    ("Arredondar ↓ (3,2)", 3, 2),
    ("Arredondar ↑ (4,3)", 4, 3),
    ("PI (ótimo inteiro)", 3, 3),
    ("Teste (0, 6)", 0, 6),
    ("Teste 2 (1,8)", 1, 8),
    ("Teste (2,4)", 2, 4),
    ("Teste (3,4)", 3, 4),
    ("Teste (2,5)", 2, 5),
    ("Teste (2,6)", 2, 6)
    
]

print(f"{'Cenário':>25} {'Max':>5} {'Pro':>5} {'Prod':>6} {'Custo':>10} {'Viável?':>10}")
print("-" * 65)
for nome, cm, gp in cenarios:
    prod = 5*cm + 4*gp
    custo = 550*cm + 97*gp
    pessoas = cm + gp
    cap = 10*cm + 8*gp
    viavel = "✅" if custo <= 2000 and pessoas == 6 and cap >= 30 else "❌"
    if pessoas != 6: viavel += f" ({pessoas:.0f} pessoas)"
    print(f"{nome:>25} {cm:>5.2f} {gp:>5.2f} {prod:>6.2f} {'R$ '+str(int(custo)):>10} {viavel:>10}")

print(f"\n⚠️  Arredondar pra baixo (3,2) = só 5 pessoas! Inviável.")
print(f"⚠️  Arredondar pra cima (4,3) = R$ 2.491! Estoura o budget.")
print(f"\n→ Arredondar NÃO funciona. Programação Inteira é o caminho.")

                  Cenário   Max   Pro   Prod      Custo    Viável?
-----------------------------------------------------------------
         PL (fracionário)  3.13  2.87  27.13    R$ 1999          ✅
       Arredondar ↓ (3,2)  3.00  2.00  23.00    R$ 1844 ❌ (5 pessoas)
       Arredondar ↑ (4,3)  4.00  3.00  32.00    R$ 2491 ❌ (7 pessoas)
       PI (ótimo inteiro)  3.00  3.00  27.00    R$ 1941          ✅
             Teste (0, 6)  0.00  6.00  24.00     R$ 582          ✅
            Teste 2 (1,8)  1.00  8.00  37.00    R$ 1326 ❌ (9 pessoas)
              Teste (2,4)  2.00  4.00  26.00    R$ 1488          ✅
              Teste (3,4)  3.00  4.00  31.00    R$ 2038 ❌ (7 pessoas)
              Teste (2,5)  2.00  5.00  30.00    R$ 1585 ❌ (7 pessoas)
              Teste (2,6)  2.00  6.00  34.00    R$ 1682 ❌ (8 pessoas)

⚠️  Arredondar pra baixo (3,2) = só 5 pessoas! Inviável.
⚠️  Arredondar pra cima (4,3) = R$ 2.491! Estoura o budget.

→ Arredondar NÃO funciona. Programação Inteira é o caminho.


---
## 4. E quando a decisão é Sim ou Não?

Às vezes a decisão não é "quantos", mas **"faço ou não faço?"**

- Invisto nesse projeto? **Sim (1) ou Não (0)**
- Abro essa filial? **Sim (1) ou Não (0)**
- Contrato essa pessoa? **Sim (1) ou Não (0)**

No PuLP:

```python
x = LpVariable("projeto", cat="Binary")  # só pode ser 0 ou 1
```

Isso é o famoso **Problema da Mochila** (Knapsack Problem).

### Problema: Seleção de Projetos

A empresa tem 8 projetos candidatos para o próximo trimestre.

| # | Projeto | Investimento (R$ mil) | Horas dev | Retorno (score 1-10) |
|---|---------|----------------------|-----------|---------------------|
| 1 | App Mobile | 150 | 400 | 8 |
| 2 | Chatbot IA | 80 | 200 | 6 |
| 3 | Dashboard BI | 60 | 120 | 5 |
| 4 | Automação RPA | 100 | 250 | 7 |
| 5 | API Integração | 40 | 80 | 4 |
| 6 | ML Previsão | 200 | 500 | 9 |
| 7 | Site Novo | 30 | 60 | 3 |
| 8 | CRM Custom | 120 | 300 | 7 |

**Restrições:**
- Budget: R$ 400 mil
- Equipe: 1.000 horas disponíveis

**Objetivo:** Maximizar o retorno total (soma dos scores).

**Decisão:** Cada projeto entra ou não entra. Não dá pra fazer "meio projeto".

In [47]:
# Dados dos projetos
projetos = [
    ("App Mobile",      150, 400, 8),
    ("Chatbot IA",       80, 200, 6),
    ("Dashboard BI",     60, 120, 5),
    ("Automação RPA",   100, 250, 7),
    ("API Integração",   40,  80, 4),
    ("ML Previsão",     200, 500, 9),
    ("Site Novo",        30,  60, 3),
    ("CRM Custom",      120, 300, 7),
]
#                       nome, custo, horas, retorno

prob = LpProblem("Selecao_Projetos", LpMaximize)

# Uma variável BINÁRIA por projeto: fazer (1) ou não fazer (0)
x = [LpVariable(f"proj_{i}", cat="Binary") for i in range(8)]

# Função objetivo: maximizar retorno total
prob += lpSum(projetos[i][3] * x[i] for i in range(8)), "Retorno"

# Restrições
prob += lpSum(projetos[i][1] * x[i] for i in range(8)) <= 400, "Budget"
prob += lpSum(projetos[i][2] * x[i] for i in range(8)) <= 1000, "Horas"

prob.solve(PULP_CBC_CMD(msg=0))

print("=" * 55)
print("  SELEÇÃO DE PROJETOS")
print("=" * 55)
budget_usado = 0
horas_usadas = 0
for i, p in enumerate(projetos):
    status = "✅ FAZER" if x[i].varValue == 1 else "   —"
    print(f"  {status}  {p[0]:<18} R${p[1]:>4}k  {p[2]:>4}h  score {p[3]}")
    if x[i].varValue == 1:
        budget_usado += p[1]
        horas_usadas += p[2]
print("=" * 55)
print(f"  Retorno total: {value(prob.objective):.0f}")
print(f"  Budget: R$ {budget_usado}k / 400k")
print(f"  Horas:  {horas_usadas} / 1000")

  SELEÇÃO DE PROJETOS
     —  App Mobile         R$ 150k   400h  score 8
  ✅ FAZER  Chatbot IA         R$  80k   200h  score 6
  ✅ FAZER  Dashboard BI       R$  60k   120h  score 5
  ✅ FAZER  Automação RPA      R$ 100k   250h  score 7
  ✅ FAZER  API Integração     R$  40k    80h  score 4
     —  ML Previsão        R$ 200k   500h  score 9
     —  Site Novo          R$  30k    60h  score 3
  ✅ FAZER  CRM Custom         R$ 120k   300h  score 7
  Retorno total: 29
  Budget: R$ 400k / 400k
  Horas:  950 / 1000


### Por que o ML Previsão (score 9) ficou de fora?

Ele custa R$ 200k e 500h — **metade** dos recursos sozinho.

O solver percebeu que é melhor fazer **5 projetos menores** (score 29) do que 2 grandes (score ~17).

Esse é o **trade-off** que a otimização revela: o projeto com maior retorno individual nem sempre entra na solução ótima.

In [40]:
# E se o gestor EXIGIR que o ML Previsão entre?
# Basta adicionar: x[5] == 1

prob2 = LpProblem("Selecao_Com_ML", LpMaximize)
x = [LpVariable(f"proj_{i}", cat="Binary") for i in range(8)]

prob2 += lpSum(projetos[i][3] * x[i] for i in range(8))
prob2 += lpSum(projetos[i][1] * x[i] for i in range(8)) <= 400
prob2 += lpSum(projetos[i][2] * x[i] for i in range(8)) <= 1000
prob2 += x[5] == 1, "ML_obrigatorio"  # <-- forçando ML Previsão

prob2.solve(PULP_CBC_CMD(msg=0))

print("COM ML Previsão obrigatório:")
print("=" * 55)
budget2 = 0
horas2 = 0
for i, p in enumerate(projetos):
    if x[i].varValue == 1:
        print(f"  ✅ {p[0]:<18} R${p[1]:>4}k  {p[2]:>4}h  score {p[3]}")
        budget2 += p[1]
        horas2 += p[2]
print(f"\n  Retorno: {value(prob2.objective):.0f} (antes era 29)")
print(f"  Budget: R$ {budget2}k / 400k")
print(f"  Horas:  {horas2} / 1000")
print(f"\n💡 Forçar o ML Previsão derruba o retorno total.")
print(f"   Esse é o tipo de insight que o gestor precisa ver.")

COM ML Previsão obrigatório:
  ✅ Dashboard BI       R$  60k   120h  score 5
  ✅ Automação RPA      R$ 100k   250h  score 7
  ✅ API Integração     R$  40k    80h  score 4
  ✅ ML Previsão        R$ 200k   500h  score 9

  Retorno: 25 (antes era 29)
  Budget: R$ 400k / 400k
  Horas:  950 / 1000

💡 Forçar o ML Previsão derruba o retorno total.
   Esse é o tipo de insight que o gestor precisa ver.


---
## 5. Análise de Sensibilidade — "E se...?"

O poder da otimização não é só achar a resposta — é **explorar cenários**.

E se o budget fosse diferente? Quais projetos entram e saem?

In [10]:
# Análise de sensibilidade: variando o budget

print(f"{'Budget (R$k)':>14} {'Projetos selecionados':<45} {'Retorno':>8}")
print("-" * 70)

for budget in [150, 200, 250, 300, 350, 400, 500, 700, 1000]:
    prob_s = LpProblem("Sensibilidade", LpMaximize)
    x = [LpVariable(f"p_{i}", cat="Binary") for i in range(8)]
    prob_s += lpSum(projetos[i][3] * x[i] for i in range(8))
    prob_s += lpSum(projetos[i][1] * x[i] for i in range(8)) <= budget
    prob_s += lpSum(projetos[i][2] * x[i] for i in range(8)) <= 1000
    prob_s.solve(PULP_CBC_CMD(msg=0))

    selecionados = [projetos[i][0] for i in range(8) if x[i].varValue == 1]
    print(f"{budget:>14} {', '.join(selecionados):<45} {value(prob_s.objective):>8.0f}")

print(f"\n💡 De 200 pra 350, cada R$ 50k a mais traz projetos novos.")
print(f"   A partir de 400, o retorno PARA de crescer. Mesmo com R$ 1 milhão!")

  Budget (R$k) Projetos selecionados                          Retorno
----------------------------------------------------------------------
           150 Chatbot IA, API Integração, Site Novo               13
           200 Dashboard BI, Automação RPA, API Integração         16


           250 Chatbot IA, Automação RPA, API Integração, Site Novo       20
           300 Chatbot IA, Dashboard BI, Automação RPA, API Integração       22
           350 Dashboard BI, Automação RPA, API Integração, Site Novo, CRM Custom       26
           400 Chatbot IA, Dashboard BI, Automação RPA, API Integração, CRM Custom       29
           500 Chatbot IA, Dashboard BI, Automação RPA, API Integração, CRM Custom       29
           700 Chatbot IA, Dashboard BI, Automação RPA, API Integração, CRM Custom       29
          1000 Chatbot IA, Dashboard BI, Automação RPA, API Integração, CRM Custom       29

💡 De 200 pra 350, cada R$ 50k a mais traz projetos novos.
   A partir de 400, o retorno PARA de crescer. Mesmo com R$ 1 milhão!


### E se eu tivesse dinheiro infinito? Conseguiria fazer todos?

In [11]:
# Quanto custaria fazer TODOS os projetos?

custo_total = sum(p[1] for p in projetos)
horas_total = sum(p[2] for p in projetos)
score_total = sum(p[3] for p in projetos)

print(f"Se fizesse TODOS os 8 projetos:")
print(f"  Custo total:  R$ {custo_total}k")
print(f"  Horas total:  {horas_total}h")
print(f"  Score total:  {score_total}")
print(f"\n  Horas disponíveis: 1.000h")
print(f"  Horas necessárias: {horas_total}h")
print(f"\n❌ IMPOSSÍVEL! Mesmo com dinheiro infinito.")
print(f"   A equipe precisaria de {horas_total}h mas só tem 1.000h.")
print(f"   O gargalo NÃO é dinheiro — é TEMPO.")
print(f"\n💡 Isso é um insight poderoso pra qualquer gestor:")
print(f"   'Chefe, não adianta aprovar mais budget.")
print(f"   'O que precisamos é contratar mais gente.'")

Se fizesse TODOS os 8 projetos:
  Custo total:  R$ 780k
  Horas total:  1910h
  Score total:  49

  Horas disponíveis: 1.000h
  Horas necessárias: 1910h

❌ IMPOSSÍVEL! Mesmo com dinheiro infinito.
   A equipe precisaria de 1910h mas só tem 1.000h.
   O gargalo NÃO é dinheiro — é TEMPO.

💡 Isso é um insight poderoso pra qualquer gestor:
   'Chefe, não adianta aprovar mais budget.
   'O que precisamos é contratar mais gente.'


---
## 6. Quem faz o quê? — Designação

**Cenário:** Uma startup tem 6 pessoas e 6 tarefas. Cada pessoa tem uma **afinidade diferente** com cada tarefa (nota 1 a 10).

**Regra:** Cada pessoa faz exatamente 1 tarefa. Cada tarefa é feita por exatamente 1 pessoa.

**Objetivo:** Maximizar a afinidade total do time.

| | Frontend | Backend | Data | DevOps | Design | QA |
|---|---|---|---|---|---|---|
| **Ana** | 7 | 4 | 3 | 2 | **9** | 4 |
| **Bruno** | 3 | 8 | **9** | 5 | 2 | 4 |
| **Carol** | 4 | 7 | **8** | 3 | 5 | 6 |
| **Diego** | 5 | 6 | 4 | 7 | 3 | **8** |
| **Eva** | 8 | 3 | 5 | 2 | **9** | 3 |
| **Fábio** | 6 | 5 | 7 | **8** | 4 | 5 |

Reparem nos **conflitos**: Ana e Eva disputam Design. Bruno, Carol e Fábio disputam Data.

Se cada um escolhesse sua preferência, teríamos conflito. O solver resolve isso.

In [12]:
pessoas = ['Ana', 'Bruno', 'Carol', 'Diego', 'Eva', 'Fábio']
tarefas = ['Frontend', 'Backend', 'Data', 'DevOps', 'Design', 'QA']

# Matriz de afinidade
afinidade = [
    [7, 4, 3, 2, 9, 4],  # Ana
    [3, 8, 9, 5, 2, 4],  # Bruno
    [4, 7, 8, 3, 5, 6],  # Carol
    [5, 6, 4, 7, 3, 8],  # Diego
    [8, 3, 5, 2, 9, 3],  # Eva
    [6, 5, 7, 8, 4, 5],  # Fábio
]

prob = LpProblem("Designacao", LpMaximize)

# Variável binária: x[i][j] = 1 se pessoa i faz tarefa j
x = [[LpVariable(f"x_{i}_{j}", cat="Binary") for j in range(6)] for i in range(6)]

# Objetivo: maximizar afinidade total
prob += lpSum(afinidade[i][j] * x[i][j] for i in range(6) for j in range(6))

# Cada pessoa faz exatamente 1 tarefa
for i in range(6):
    prob += lpSum(x[i][j] for j in range(6)) == 1, f"Pessoa_{pessoas[i]}"

# Cada tarefa é feita por exatamente 1 pessoa
for j in range(6):
    prob += lpSum(x[i][j] for i in range(6)) == 1, f"Tarefa_{tarefas[j]}"

prob.solve(PULP_CBC_CMD(msg=0))

print("=" * 50)
print("  ALOCAÇÃO ÓTIMA")
print("=" * 50)
for i in range(6):
    for j in range(6):
        if x[i][j].varValue == 1:
            melhor = max(afinidade[i])
            flag = " ⚡ (não é sua melhor!)" if afinidade[i][j] < melhor else " ⭐"
            print(f"  {pessoas[i]:<8} → {tarefas[j]:<10} afinidade {afinidade[i][j]}{flag}")
print(f"\n  Afinidade total: {value(prob.objective):.0f}")
print(f"\n💡 Bruno e Eva NÃO ficaram com sua tarefa preferida.")
print(f"   Mas o resultado GLOBAL é o melhor possível.")

  ALOCAÇÃO ÓTIMA
  Ana      → Design     afinidade 9 ⭐
  Bruno    → Backend    afinidade 8 ⚡ (não é sua melhor!)
  Carol    → Data       afinidade 8 ⭐
  Diego    → QA         afinidade 8 ⭐
  Eva      → Frontend   afinidade 8 ⚡ (não é sua melhor!)
  Fábio    → DevOps     afinidade 8 ⭐

  Afinidade total: 49

💡 Bruno e Eva NÃO ficaram com sua tarefa preferida.
   Mas o resultado GLOBAL é o melhor possível.


In [13]:
# E se cada um escolhesse sua preferência (abordagem "sem otimização")?

print("SE CADA UM ESCOLHESSE SUA PREFERÊNCIA:")
print("=" * 50)
usadas = {}
conflitos = 0
for i in range(6):
    melhor_j = max(range(6), key=lambda j: afinidade[i][j])
    if tarefas[melhor_j] in usadas:
        print(f"  {pessoas[i]:<8} → {tarefas[melhor_j]:<10} ❌ CONFLITO com {usadas[tarefas[melhor_j]]}!")
        conflitos += 1
    else:
        usadas[tarefas[melhor_j]] = pessoas[i]
        print(f"  {pessoas[i]:<8} → {tarefas[melhor_j]:<10} afinidade {afinidade[i][melhor_j]}")

print(f"\n  {conflitos} conflito(s)! Sem otimização, alguém fica sem tarefa.")

SE CADA UM ESCOLHESSE SUA PREFERÊNCIA:
  Ana      → Design     afinidade 9
  Bruno    → Data       afinidade 9
  Carol    → Data       ❌ CONFLITO com Bruno!
  Diego    → QA         afinidade 8
  Eva      → Design     ❌ CONFLITO com Ana!
  Fábio    → DevOps     afinidade 8

  2 conflito(s)! Sem otimização, alguém fica sem tarefa.


---
## 7. O que o solver faz por baixo? — Branch-and-Bound

Quando você escreve `cat="Integer"`, o PuLP não testa todas as combinações.

Ele usa **Branch-and-Bound**:

1. **Relaxa** — resolve como PL (sem restrição de inteiro). Isso dá o "teto".
2. **Branch** — pega uma variável fracionária e cria dois subproblemas: x ≤ 3 e x ≥ 4
3. **Bound** — se um ramo já não pode melhorar o melhor resultado encontrado, **poda** (ignora)
4. **Repete** até achar a solução inteira ótima

É como uma árvore de decisão que vai cortando caminhos ruins.

### Vamos ver isso acontecendo:

In [14]:
# Simulando Branch-and-Bound no problema dos Planos de IA

def resolver_relaxado(extra_constraints=None):
    """Resolve a versão relaxada (PL) com restrições extras."""
    p = LpProblem("BB", LpMaximize)
    x1 = LpVariable("Codex_Go", lowBound=0)
    x2 = LpVariable("Codex_Plus", lowBound=0)
    x3 = LpVariable("Claude_Pro", lowBound=0)
    x4 = LpVariable("Claude_Max", lowBound=0)
    x5 = LpVariable("Gemini_Plus", lowBound=0)
    x6 = LpVariable("Gemini_Pro", lowBound=0)
    p += 3*x1 + 4*x2 + 4*x3 + 5*x4 + 3*x5 + 4*x6
    p += 45*x1+110*x2+110*x3+550*x4+25*x5+97*x6 <= 2000
    p += x1+x2+x3+x4+x5+x6 == 6
    p += 2*x1+5*x2+7*x3+10*x4+3*x5+8*x6 >= 30
    if extra_constraints:
        for var, op, val in extra_constraints:
            v = {'x4': x4, 'x6': x6}[var]
            if op == '<=': p += v <= val
            else: p += v >= val
    p.solve(PULP_CBC_CMD(msg=0))
    if p.status != 1: return None
    return {v.name: v.varValue for v in p.variables()}, value(p.objective)

# Nó 0: Relaxação linear (sem restrição de inteiro)
r0, z0 = resolver_relaxado()
print(f"Nó 0 (relaxação): Z = {z0:.2f}")
print(f"  Claude_Max = {r0['Claude_Max']:.4f}, Gemini_Pro = {r0['Gemini_Pro']:.4f}")
print(f"  → Claude_Max é fracionário! Branch em Claude_Max.")

# Nó 1: Claude_Max <= 3
r1, z1 = resolver_relaxado([('x4', '<=', 3)])
print(f"\nNó 1 (Claude_Max ≤ 3): Z = {z1:.2f}")
print(f"  Claude_Max = {r1['Claude_Max']:.4f}, Gemini_Pro = {r1['Gemini_Pro']:.4f}")

# Nó 2: Claude_Max >= 4
r2 = resolver_relaxado([('x4', '>=', 4)])
if r2:
    print(f"\nNó 2 (Claude_Max ≥ 4): Z = {r2[1]:.2f}")
else:
    print(f"\nNó 2 (Claude_Max ≥ 4): INVIÁVEL! ✂️ Podado.")
    print(f"  4 × R$550 = R$2.200 > R$2.000. Não cabe no budget.")

# Nó 1 deu inteiro?
tudo_inteiro = all(abs(v - round(v)) < 0.001 for v in r1.values())
if tudo_inteiro:
    print(f"\n✅ Nó 1 deu solução inteira! Z = {z1:.0f}")
    print(f"   Como o outro ramo foi podado, essa é a solução ÓTIMA.")
else:
    print(f"\n   Nó 1 ainda tem fracionários. Continuar branching...")

Nó 0 (relaxação): Z = 27.13
  Claude_Max = 3.1302, Gemini_Pro = 2.8698
  → Claude_Max é fracionário! Branch em Claude_Max.



Nó 1 (Claude_Max ≤ 3): Z = 27.00
  Claude_Max = 3.0000, Gemini_Pro = 3.0000

Nó 2 (Claude_Max ≥ 4): INVIÁVEL! ✂️ Podado.
  4 × R$550 = R$2.200 > R$2.000. Não cabe no budget.

✅ Nó 1 deu solução inteira! Z = 27
   Como o outro ramo foi podado, essa é a solução ÓTIMA.


---
## 8. Por que não testar tudo? — P vs NP

**Por que não testar todas as combinações?**

Porque o número de combinações **explode**:

In [15]:
import math

print("Problema da Mochila (8 projetos, cada um sim/não):")
print(f"  Combinações: 2⁸ = {2**8}")
print(f"  Um computador resolve em milissegundos.\n")

print("E se fossem 30 projetos?")
print(f"  Combinações: 2³⁰ = {2**30:,}")
print(f"  Ainda vai, mas já demora.\n")

print("E 100 projetos?")
print(f"  Combinações: 2¹⁰⁰ = {2**100:,}")
print(f"  Mais que o número de átomos no universo.\n")

print("Caixeiro Viajante (n cidades):")
for n in [5, 10, 15, 20, 25]:
    rotas = math.factorial(n-1) // 2  # rotas únicas em TSP simétrico
    print(f"  {n:>3} cidades → {rotas:>20,} rotas")

print(f"\n💡 É por isso que Branch-and-Bound é genial:")
print(f"   ele PODA ramos inteiros sem testar.")
print(f"   Em vez de 2¹⁰⁰ combinações, testa talvez umas 1.000.")

Problema da Mochila (8 projetos, cada um sim/não):
  Combinações: 2⁸ = 256
  Um computador resolve em milissegundos.

E se fossem 30 projetos?
  Combinações: 2³⁰ = 1,073,741,824
  Ainda vai, mas já demora.

E 100 projetos?
  Combinações: 2¹⁰⁰ = 1,267,650,600,228,229,401,496,703,205,376
  Mais que o número de átomos no universo.

Caixeiro Viajante (n cidades):
    5 cidades →                   12 rotas
   10 cidades →              181,440 rotas
   15 cidades →       43,589,145,600 rotas
   20 cidades → 60,822,550,204,416,000 rotas
   25 cidades → 310,224,200,866,619,719,680,000 rotas

💡 É por isso que Branch-and-Bound é genial:
   ele PODA ramos inteiros sem testar.
   Em vez de 2¹⁰⁰ combinações, testa talvez umas 1.000.


### O que é P vs NP?

- **P** = problemas que um computador resolve **rápido** (em tempo polinomial). Ex: ordenar uma lista, PL (Simplex).
- **NP** = problemas onde **verificar** uma solução é rápido, mas **encontrar** pode ser lento. Ex: Mochila, TSP, Designação.
- **NP-difícil** = os mais duros do NP. Se alguém provar que P = NP, ganha US$ 1 milhão (Millennium Prize).

**Na prática:** pra problemas grandes, usamos heurísticas (Aula 3) — soluções "boas o suficiente" em tempo aceitável.

---
## 9. Caixeiro Viajante — O problema mais famoso

Um vendedor precisa visitar N cidades e voltar à origem. **Qual a rota mais curta?**

Vamos resolver com PuLP para 6 cidades.

In [16]:
# Cidades e distâncias (km)
cidades = ['Fortaleza', 'Recife', 'Salvador', 'Natal', 'João Pessoa', 'Maceió']
n = len(cidades)

# Matriz de distâncias (km, aproximadas)
dist = [
    [  0, 800, 1300, 530, 690, 1050],  # Fortaleza
    [800,   0,  840, 300, 120,  260],  # Recife
    [1300, 840,  0, 1100, 950,  630],  # Salvador
    [530, 300, 1100,  0, 185,  530],  # Natal
    [690, 120,  950, 185,  0,  390],  # João Pessoa
    [1050, 260, 630, 530, 390,   0],  # Maceió
]

prob = LpProblem("TSP", LpMinimize)

# x[i][j] = 1 se viajamos de cidade i para cidade j
x = [[LpVariable(f"x_{i}_{j}", cat="Binary") if i != j else None
       for j in range(n)] for i in range(n)]

# Variáveis auxiliares para eliminar sub-rotas (MTZ)
u = [LpVariable(f"u_{i}", lowBound=1, upBound=n-1) for i in range(n)]

# Objetivo: minimizar distância total
prob += lpSum(dist[i][j] * x[i][j] for i in range(n) for j in range(n) if i != j)

# Cada cidade é visitada exatamente 1 vez (entrada)
for j in range(n):
    prob += lpSum(x[i][j] for i in range(n) if i != j) == 1

# Cada cidade é saída exatamente 1 vez
for i in range(n):
    prob += lpSum(x[i][j] for j in range(n) if i != j) == 1

# Restrições MTZ (eliminam sub-rotas)
for i in range(1, n):
    for j in range(1, n):
        if i != j:
            prob += u[i] - u[j] + n * x[i][j] <= n - 1

prob.solve(PULP_CBC_CMD(msg=0))

# Reconstruir a rota
print("=" * 50)
print("  ROTA ÓTIMA — Caixeiro Viajante")
print("=" * 50)
rota = [0]
atual = 0
dist_total = 0
for _ in range(n - 1):
    for j in range(n):
        if j != atual and x[atual][j] is not None and x[atual][j].varValue == 1:
            dist_total += dist[atual][j]
            print(f"  {cidades[atual]:>15} → {cidades[j]:<15} {dist[atual][j]:>5} km")
            rota.append(j)
            atual = j
            break
dist_total += dist[atual][0]
print(f"  {cidades[atual]:>15} → {cidades[0]:<15} {dist[atual][0]:>5} km")
print(f"{'':>15}{'':>20}{'─' * 9}")
print(f"  {'TOTAL':>15} {'':>18} {dist_total:>5} km")
print("=" * 50)

  ROTA ÓTIMA — Caixeiro Viajante
        Fortaleza → Natal             530 km
            Natal → João Pessoa       185 km
      João Pessoa → Recife            120 km
           Recife → Maceió            260 km
           Maceió → Salvador          630 km
         Salvador → Fortaleza        1300 km
                                   ─────────
            TOTAL                     3025 km


---
## 10. Onde isso aparece na IA?

Vocês acabaram de ver Mochila, Designação, TSP. Parecem problemas "clássicos"?

**Esses problemas aparecem o tempo todo em IA moderna.** Alguns exemplos:

### Machine Learning e Deep Learning

| Problema de IA | Formulação como PO |
|---|---|
| **Treinar uma rede neural** | Minimizar a loss function (otimização contínua → gradient descent) |
| **Seleção de features** | Mochila binária: quais variáveis usar? Cada uma tem "peso" (complexidade) e "valor" (informação) |
| **Hyperparameter tuning** | Otimização combinatória: qual learning rate, batch size, nº de camadas? (Aula 3 com Optuna) |
| **Neural Architecture Search (NAS)** | Selecionar camadas e conexões de uma rede — variáveis binárias! |

### Processamento de Linguagem Natural (PLN)

| Problema de PLN | Formulação como PO |
|---|---|
| **Alocação de tokens em LLMs** | Mochila! Context window = 128k tokens. Quanto pro system prompt? Quanto pro RAG? Quanto pro histórico? Maximizar qualidade da resposta. |
| **Beam Search (geração de texto)** | Branch-and-Bound! A cada token gerado, explora K caminhos, poda os piores, mantém os melhores |
| **Chunking em RAG** | Como dividir documentos em pedaços pra maximizar relevância dado limite de tokens? Otimização combinatória |
| **Roteamento de agentes de IA** | Designação! Se tenho 5 agentes especializados e chega uma mensagem, qual agente atende? |
| **RLHF (alinhamento de LLMs)** | Maximizar reward com restrição: o modelo não pode se distanciar demais do original (KL divergence ≤ limite) |

### Um exemplo concreto: Beam Search = Branch-and-Bound

In [17]:
# Beam Search simplificado — como um LLM gera texto

import random
random.seed(42)

vocabulario = {
    "O":     {"gato": 0.4, "cachorro": 0.3, "pássaro": 0.2, "rato": 0.1},
    "gato":  {"dormiu": 0.5, "comeu": 0.3, "fugiu": 0.2},
    "cachorro": {"latiu": 0.5, "correu": 0.3, "dormiu": 0.2},
    "pássaro":  {"voou": 0.6, "cantou": 0.3, "dormiu": 0.1},
    "rato":     {"fugiu": 0.5, "comeu": 0.3, "escondeu": 0.2},
}

def beam_search(inicio, beam_width=2, max_tokens=3):
    """Gera texto usando Beam Search (Branch-and-Bound simplificado)."""
    # Começa com K=beam_width caminhos
    caminhos = [([inicio], 1.0)]

    print(f"Beam Search (K={beam_width}):\n")

    for passo in range(max_tokens):
        todos_candidatos = []

        for seq, prob in caminhos:
            ultima = seq[-1]
            if ultima not in vocabulario:
                todos_candidatos.append((seq, prob))
                continue
            # BRANCH: expande cada caminho com todas as palavras possíveis
            for palavra, p in vocabulario[ultima].items():
                todos_candidatos.append((seq + [palavra], prob * p))

        # BOUND + PODA: mantém apenas os K melhores
        todos_candidatos.sort(key=lambda x: x[1], reverse=True)
        caminhos = todos_candidatos[:beam_width]

        print(f"  Passo {passo+1}:")
        for seq, prob in caminhos:
            print(f"    {' '.join(seq):<30} prob = {prob:.3f}")

        # Mostra quantos foram podados
        podados = len(todos_candidatos) - beam_width
        if podados > 0:
            print(f"    ✂️  {podados} caminhos podados")
        print()

    melhor = caminhos[0]
    print(f"Resultado: \"{' '.join(melhor[0])}\" (prob = {melhor[1]:.4f})")
    print(f"\n💡 Isso é exatamente Branch-and-Bound:")
    print(f"   BRANCH = gerar todas as próximas palavras possíveis")
    print(f"   BOUND  = probabilidade acumulada")
    print(f"   PODA   = descartar caminhos com probabilidade baixa")

beam_search("O", beam_width=2, max_tokens=3)

Beam Search (K=2):

  Passo 1:
    O gato                         prob = 0.400
    O cachorro                     prob = 0.300
    ✂️  2 caminhos podados

  Passo 2:
    O gato dormiu                  prob = 0.200
    O cachorro latiu               prob = 0.150
    ✂️  4 caminhos podados

  Passo 3:
    O gato dormiu                  prob = 0.200
    O cachorro latiu               prob = 0.150

Resultado: "O gato dormiu" (prob = 0.2000)

💡 Isso é exatamente Branch-and-Bound:
   BRANCH = gerar todas as próximas palavras possíveis
   BOUND  = probabilidade acumulada
   PODA   = descartar caminhos com probabilidade baixa


### Outro exemplo: Mochila de Tokens em RAG

Quando um LLM recebe uma pergunta e busca contexto em documentos (**RAG** — Retrieval Augmented Generation), ele tem um **limite de tokens** na context window.

Cada chunk tem um **tamanho** (tokens) e uma **relevância** (score de similaridade).

**Isso é literalmente o Problema da Mochila!**

Exemplo real: um assistente de IA para clínicas médicas precisa responder uma dúvida do paciente. Tem 8 chunks disponíveis mas só 3.500 tokens de espaço.

In [18]:
# Mochila de Tokens — seleção de chunks para RAG

chunks = [
    ("Histórico do paciente",       1200, 0.95),
    ("Bula do medicamento",         2000, 0.80),
    ("Protocolo de atendimento",     800, 0.92),
    ("Últimas consultas",           1100, 0.93),
    ("Resultados de exames",         900, 0.88),
    ("Agenda disponível",            700, 0.85),
    ("FAQ da clínica",               500, 0.35),
    ("Dados cadastrais",             300, 0.20),
]
#                                   tokens  relevância

# Context window disponível para RAG
MAX_TOKENS = 3500

prob = LpProblem("RAG_Chunks", LpMaximize)
x = [LpVariable(f"chunk_{i}", cat="Binary") for i in range(len(chunks))]

# Maximizar relevância total
prob += lpSum(chunks[i][2] * x[i] for i in range(len(chunks)))

# Não estourar a context window
prob += lpSum(chunks[i][1] * x[i] for i in range(len(chunks))) <= MAX_TOKENS

prob.solve(PULP_CBC_CMD(msg=0))

print("=" * 60)
print("  SELEÇÃO DE CHUNKS PARA RAG")
print(f"  Context window disponível: {MAX_TOKENS} tokens")
print("=" * 60)
tokens_usados = 0
for i, c in enumerate(chunks):
    if x[i].varValue == 1:
        print(f"  ✅ {c[0]:<28} {c[1]:>5} tokens  relevância {c[2]:.2f}")
        tokens_usados += c[1]
    else:
        print(f"     {c[0]:<28} {c[1]:>5} tokens  relevância {c[2]:.2f}")
print(f"\n  Tokens usados: {tokens_usados} / {MAX_TOKENS}")
print(f"  Relevância total: {value(prob.objective):.2f}")
print(f"\n💡 'Histórico do paciente' tem a MAIOR relevância (0.95)")
print(f"   mas ficou de fora! Com 1.200 tokens, não cabe junto com os outros.")
print(f"   'Bula do medicamento' também ficou de fora — 2.000 tokens é demais.")
print(f"\n   Exatamente como o ML Previsão na seleção de projetos:")
print(f"   o item mais valioso individualmente nem sempre entra na solução ótima.")

  SELEÇÃO DE CHUNKS PARA RAG
  Context window disponível: 3500 tokens
     Histórico do paciente         1200 tokens  relevância 0.95
     Bula do medicamento           2000 tokens  relevância 0.80
  ✅ Protocolo de atendimento       800 tokens  relevância 0.92
  ✅ Últimas consultas             1100 tokens  relevância 0.93
  ✅ Resultados de exames           900 tokens  relevância 0.88
  ✅ Agenda disponível              700 tokens  relevância 0.85
     FAQ da clínica                 500 tokens  relevância 0.35
     Dados cadastrais               300 tokens  relevância 0.20

  Tokens usados: 3500 / 3500
  Relevância total: 3.58

💡 'Histórico do paciente' tem a MAIOR relevância (0.95)
   mas ficou de fora! Com 1.200 tokens, não cabe junto com os outros.
   'Bula do medicamento' também ficou de fora — 2.000 tokens é demais.

   Exatamente como o ML Previsão na seleção de projetos:
   o item mais valioso individualmente nem sempre entra na solução ótima.


---
## 11. Desafio — IA vs Programação Inteira 🤖⚔️

Será que o ChatGPT consegue resolver um problema de otimização melhor que o PuLP?

**O problema:** Vocês têm **R$ 550** e **40 horas** disponíveis no mês. Quais cursos fazer para maximizar a soma das notas?

| # | Curso | Preço (R$) | Horas | Nota |
|---|-------|-----------|-------|------|
| 1 | Excel Básico | 29 | 8 | 3.0 |
| 2 | Git Básico | 35 | 7 | 3.5 |
| 3 | Scrum/Kanban | 39 | 6 | 4.0 |
| 4 | Markdown/Docs | 19 | 5 | 2.5 |
| 5 | HTML/CSS | 25 | 9 | 3.0 |
| 6 | SQL Fundamentals | 49 | 4 | 8.0 |
| 7 | Prompt Engineering | 39 | 3 | 7.0 |
| 8 | Python para Dados | 99 | 8 | 9.0 |
| 9 | Estatística Aplicada | 89 | 7 | 8.5 |
| 10 | Power BI | 79 | 6 | 7.0 |
| 11 | API REST | 69 | 5 | 6.5 |
| 12 | Docker Essentials | 89 | 6 | 7.5 |
| 13 | React Básico | 79 | 8 | 5.5 |
| 14 | Tableau | 69 | 5 | 6.0 |
| 15 | Linux CLI | 45 | 4 | 7.5 |
| 16 | Machine Learning | 199 | 8 | 9.5 |
| 17 | Deep Learning | 249 | 10 | 9.0 |
| 18 | NLP Transformers | 189 | 7 | 9.2 |
| 19 | MLOps Pipeline | 179 | 7 | 8.8 |
| 20 | Cloud AWS | 199 | 9 | 8.0 |
| 21 | Data Engineering | 219 | 9 | 8.5 |
| 22 | LLM Fine-tuning | 249 | 10 | 9.8 |
| 23 | Spark Big Data | 149 | 6 | 8.2 |
| 24 | Computer Vision | 159 | 8 | 7.8 |
| 25 | Cybersecurity | 139 | 6 | 7.0 |

### Passo 1: Perguntem ao ChatGPT

Copiem a tabela e peçam:

> "Tenho R$ 550 e 40 horas. Quais cursos selecionar para maximizar a soma das notas? Respeite ambas as restrições. Me dê a lista exata e a soma total das notas."

**Anotem a resposta e a nota total que o ChatGPT deu.**

### Passo 2: Rodem o PuLP

In [19]:
import time

cursos = [
    ("Excel Básico",           29,   8, 3.0),
    ("Git Básico",             35,   7, 3.5),
    ("Scrum/Kanban",           39,   6, 4.0),
    ("Markdown/Docs",          19,   5, 2.5),
    ("HTML/CSS",               25,   9, 3.0),
    ("SQL Fundamentals",       49,   4, 8.0),
    ("Prompt Engineering",     39,   3, 7.0),
    ("Python para Dados",      99,   8, 9.0),
    ("Estatística Aplicada",   89,   7, 8.5),
    ("Power BI",               79,   6, 7.0),
    ("API REST",               69,   5, 6.5),
    ("Docker Essentials",      89,   6, 7.5),
    ("React Básico",           79,   8, 5.5),
    ("Tableau",                69,   5, 6.0),
    ("Linux CLI",              45,   4, 7.5),
    ("Machine Learning",      199,   8, 9.5),
    ("Deep Learning",         249,  10, 9.0),
    ("NLP Transformers",      189,   7, 9.2),
    ("MLOps Pipeline",        179,   7, 8.8),
    ("Cloud AWS",             199,   9, 8.0),
    ("Data Engineering",      219,   9, 8.5),
    ("LLM Fine-tuning",      249,  10, 9.8),
    ("Spark Big Data",        149,   6, 8.2),
    ("Computer Vision",       159,   8, 7.8),
    ("Cybersecurity",         139,   6, 7.0),
]

BUDGET = 550
HORAS = 40

prob = LpProblem("Cursos", LpMaximize)
n = len(cursos)
x = [LpVariable(f"c_{i}", cat="Binary") for i in range(n)]

prob += lpSum(cursos[i][3] * x[i] for i in range(n)), "Nota_total"
prob += lpSum(cursos[i][1] * x[i] for i in range(n)) <= BUDGET, "Budget"
prob += lpSum(cursos[i][2] * x[i] for i in range(n)) <= HORAS, "Horas"

t0 = time.time()
prob.solve(PULP_CBC_CMD(msg=0))
t1 = time.time()

print("=" * 55)
print("  SOLUÇÃO ÓTIMA — Programação Inteira")
print("=" * 55)
custo_total = 0
horas_total = 0
for i, c in enumerate(cursos):
    if x[i].varValue == 1:
        print(f"  ✅ {c[0]:<25} R${c[1]:>4}  {c[2]:>3}h  nota {c[3]}")
        custo_total += c[1]
        horas_total += c[2]
print(f"\n  Custo: R$ {custo_total} / {BUDGET}")
print(f"  Horas: {horas_total} / {HORAS}")
print(f"  Nota total: {value(prob.objective):.1f}")
print(f"  Tempo: {(t1-t0)*1000:.0f} ms")
print(f"  Combinações possíveis: 2²⁵ = {2**n:,}")
print("=" * 55)
print(f"\n🤔 O ChatGPT chegou nesse resultado?")
print(f"\n   Com 25 cursos, são 33 milhões de combinações.")
print(f"   LLMs tentam heurísticas (pegar os melhores, melhor custo-benefício).")
print(f"   Nenhuma heurística simples acha o ótimo aqui.")
print(f"   O PuLP achou em milissegundos usando Branch-and-Bound.")

  SOLUÇÃO ÓTIMA — Programação Inteira
  ✅ SQL Fundamentals          R$  49    4h  nota 8.0
  ✅ Prompt Engineering        R$  39    3h  nota 7.0
  ✅ Estatística Aplicada      R$  89    7h  nota 8.5
  ✅ Power BI                  R$  79    6h  nota 7.0
  ✅ API REST                  R$  69    5h  nota 6.5
  ✅ Docker Essentials         R$  89    6h  nota 7.5
  ✅ Tableau                   R$  69    5h  nota 6.0
  ✅ Linux CLI                 R$  45    4h  nota 7.5

  Custo: R$ 528 / 550
  Horas: 40 / 40
  Nota total: 58.0
  Tempo: 25 ms
  Combinações possíveis: 2²⁵ = 33,554,432

🤔 O ChatGPT chegou nesse resultado?

   Com 25 cursos, são 33 milhões de combinações.
   LLMs tentam heurísticas (pegar os melhores, melhor custo-benefício).
   Nenhuma heurística simples acha o ótimo aqui.
   O PuLP achou em milissegundos usando Branch-and-Bound.


---
## Resumo

| Conceito | PuLP | Quando usar |
|----------|------|-------------|
| PL (fracionário) | `LpVariable("x", lowBound=0)` | Quantidades contínuas (budget, peso) |
| PI (inteiro) | `cat="Integer"` | Unidades inteiras (pessoas, licenças, máquinas) |
| Binário | `cat="Binary"` | Sim/Não (fazer projeto, abrir filial) |
| Designação | Binário + restrições == 1 | Alocar pessoas a tarefas |
| Mochila | Binário + restrição de capacidade | Selecionar itens com limite |
| TSP | Binário + MTZ | Encontrar rota mais curta |

**O que vimos hoje:**
1. `cat="Integer"` resolve o problema fracionário da Aula 1
2. Arredondar NÃO funciona — pode ser inviável ou subótimo
3. Variáveis binárias modelam decisões Sim/Não
4. Branch-and-Bound é o algoritmo por trás do solver
5. Problemas combinatórios explodem exponencialmente (P vs NP)
6. **PO está em tudo na IA:** Beam Search = B&B, RAG = Mochila, RLHF = otimização com restrição

**Próxima aula:** Quando o problema é grande demais até pro Branch-and-Bound → Heurísticas e Metaheurísticas!